# to do

create an MLP for the global features as exercises limb amplitude time some fft fearures and others 
concatanate it to the cNN

In [2]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 




from utilities.model import CNNmodel_base, CNNMLPModel
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [3]:
import pandas as pd

In [4]:
df=pd.read_csv(DATA_DIR / 'processed_data.csv')

In [5]:
limb_map = {
    "e1": 0, #leg
    "e2": 1, #arm
    "e3": 0,
    "e4": 0,
    "e5": 0,
    "e6": 1,
    "e7": 1,
    "e8": 1
}

df["limb"] = df["exercise"].map(limb_map)
df['limb'] = df['limb'].astype(np.float32)

In [6]:
exercise_dummies = pd.get_dummies(df['exercise'], prefix='exercise').astype(np.float32)
df = pd.concat([df, exercise_dummies], axis=1)

In [7]:
exercises = ['e1','e2','e3','e4','e5','e6','e7','e8']
subject=['s1', 's2', 's3','s4', 's5']
labels=['correct', 'low_amplitude', 'fast']

for s in  subject:
    for e in exercises:
        for l in labels:


            df_r=df[(df['subject'] == s) & (df['exercise'] == e) & (df['label']==l)]

            
            duration = df_r['time index'].max() - df_r['time index'].min()

            repetition_duration= duration /10

            print(repetition_duration, duration, l)

            

202.8 2028 correct
192.3 1923 low_amplitude
84.2 842 fast
192.8 1928 correct
189.3 1893 low_amplitude
125.9 1259 fast
197.5 1975 correct
202.4 2024 low_amplitude
109.1 1091 fast
204.2 2042 correct
173.6 1736 low_amplitude
101.4 1014 fast
182.1 1821 correct
185.7 1857 low_amplitude
91.2 912 fast
237.1 2371 correct
189.3 1893 low_amplitude
110.4 1104 fast
224.8 2248 correct
198.4 1984 low_amplitude
104.9 1049 fast
180.9 1809 correct
191.7 1917 low_amplitude
84.5 845 fast
157.8 1578 correct
147.5 1475 low_amplitude
82.4 824 fast
184.3 1843 correct
153.7 1537 low_amplitude
100.8 1008 fast
171.5 1715 correct
136.7 1367 low_amplitude
82.9 829 fast
146.2 1462 correct
124.4 1244 low_amplitude
84.1 841 fast
139.1 1391 correct
116.9 1169 low_amplitude
81.5 815 fast
144.3 1443 correct
120.0 1200 low_amplitude
78.4 784 fast
131.8 1318 correct
108.9 1089 low_amplitude
77.5 775 fast
120.4 1204 correct
106.8 1068 low_amplitude
74.6 746 fast
164.3 1643 correct
149.3 1493 low_amplitude
85.8 858 fast
17

In [8]:
inputs_seq = [
    'acc_mag_u1', 'gyr_mag_u1', 'mag_mag_u1',
    'acc_mag_u2', 'gyr_mag_u2', 'mag_mag_u2',
    'acc_mag_u3', 'gyr_mag_u3', 'mag_mag_u3',
    'acc_mag_u4', 'gyr_mag_u4', 'mag_mag_u4',
    'acc_mag_u5', 'gyr_mag_u5', 'mag_mag_u5'
]

inputs_global = [
    'limb',
    'exercise_e1', 'exercise_e2', 'exercise_e3', 'exercise_e4',
    'exercise_e5', 'exercise_e6', 'exercise_e7', 'exercise_e8'
]

In [9]:
label_to_idx = {
    'correct': 0,
    'low_amplitude': 1,
    'fast': 2
}

all_sequences = []
all_labels = []
all_subjects = []   
all_exercises = []  
all_global_features=[]


for s in subject:
    for e in exercises:
        for l in labels:

            df_r = df[
                (df['subject'] == s) &
                (df['exercise'] == e) &
                (df['label'] == l)
            ].copy()

            df_r = df_r.sort_values('time index')

            duration = df_r['time index'].max() - df_r['time index'].min()
            rep_len = int(duration / 10)

            for i in range(10):
                rep = df_r.iloc[i * rep_len:(i + 1) * rep_len]

                X_rep = rep[inputs_seq].to_numpy()              # (time, seq_features)
                g_rep = torch.tensor(rep[inputs_global].iloc[0].to_numpy(), dtype=torch.float32)  # (global_features,)

                all_sequences.append(X_rep)
                all_global_features.append(g_rep)
                all_labels.append(label_to_idx[l])
                all_subjects.append(s)
                all_exercises.append(e)

max_len = max(seq.shape[0] for seq in all_sequences)

padded = []

for seq in all_sequences:
    seq = torch.tensor(seq, dtype=torch.float32)  # (time, features)
    seq = seq.transpose(0, 1)                     # (features, time)

    pad_len = max_len - seq.shape[1]
    seq = F.pad(seq, (0, pad_len))

    padded.append(seq)

X_seq = torch.stack(padded)  # (N, features, max_len)
X_glob=torch.stack(all_global_features)
y = torch.tensor(all_labels, dtype=torch.long)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [11]:
all_subjects = np.array(all_subjects)
unique_subjects = np.unique(all_subjects)
fold_results = []

print(type(all_subjects))
print(np.array(all_subjects).shape)
print(all_subjects[:10] if hasattr(all_subjects, "__len__") else all_subjects)
'''print("len(X) =", len(X))
print("len(y) =", len(y))
print("len(all_subjects) =", len(all_subjects) if hasattr(all_subjects, "__len__") else "NO LEN")'''


<class 'numpy.ndarray'>
(1200,)
['s1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1']


'print("len(X) =", len(X))\nprint("len(y) =", len(y))\nprint("len(all_subjects) =", len(all_subjects) if hasattr(all_subjects, "__len__") else "NO LEN")'

In [12]:


for val_subject in unique_subjects:
    print(f"\n========== Fold: leaving out {val_subject} ==========")

    train_indices = [i for i, s in enumerate(all_subjects) if s != val_subject]
    val_indices = [i for i, s in enumerate(all_subjects) if s == val_subject]

    X_seq_train = X_seq[train_indices]
    X_seq_val   = X_seq[val_indices]

    X_glob_train = X_glob[train_indices]
    X_glob_val   = X_glob[val_indices]

    y_train = y[train_indices]
    y_val   = y[val_indices]

    '''  train_mean = X_train.mean(dim=(0,2), keepdim=True)
    train_std = X_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_train = (X_train - train_mean) / train_std
    X_val = (X_val - train_mean) / train_std'''




    
    train_dataset = TensorDataset(
    X_seq_train,
    X_glob_train,
    y_train
)

    val_dataset = TensorDataset(
    X_seq_val,
    X_glob_val,
    y_val
)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # fresh model for each fold
    model = CNNMLPModel(input_dim_seq=15,input_dim_global=9, num_classes=3).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_loss = float("inf")
    best_preds = None
    best_labels = None

    for epoch in range(20):
        # TRAIN
        model.train()
        running_loss = 0.0

        for x_seq_b, x_glob_b, y_b in train_loader:
            x_seq_b = x_seq_b.to(device)
            x_glob_b = x_glob_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()

            outputs = model(x_seq_b, x_glob_b)
            loss = criterion(outputs, y_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x_seq_b.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # VALIDATION
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for x_seq_b, x_glob_b, y_b in val_loader:
                x_seq_b = x_seq_b.to(device)
                x_glob_b = x_glob_b.to(device)
                y_b = y_b.to(device)

                outputs = model(x_seq_b, x_glob_b)
                loss = criterion(outputs, y_b)

                val_running_loss += loss.item() * x_seq_b.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_b.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        print(
            f"Fold {val_subject} | Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        # keep best epoch for this fold
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = np.array(all_preds)
            best_labels = np.array(all_labels)

    # fold-level metrics
    fold_acc = accuracy_score(best_labels, best_preds)
    fold_f1 = f1_score(best_labels, best_preds, average="macro")
    fold_cm = confusion_matrix(best_labels, best_preds)

    fold_results.append({
        "subject": val_subject,
        "val_loss": best_val_loss,
        "accuracy": fold_acc,
        "macro_f1": fold_f1,
        "confusion_matrix": fold_cm
    })

    


========== Fold: leaving out s1 ==========
Fold s1 | Epoch 1 | Train Loss: 1.0444 | Val Loss: 0.9582 | Val Acc: 0.6000
Fold s1 | Epoch 2 | Train Loss: 0.9256 | Val Loss: 0.8423 | Val Acc: 0.5833
Fold s1 | Epoch 3 | Train Loss: 0.8157 | Val Loss: 0.7444 | Val Acc: 0.6458
Fold s1 | Epoch 4 | Train Loss: 0.7373 | Val Loss: 0.8553 | Val Acc: 0.4375
Fold s1 | Epoch 5 | Train Loss: 0.6749 | Val Loss: 0.7104 | Val Acc: 0.7167
Fold s1 | Epoch 6 | Train Loss: 0.6239 | Val Loss: 0.6059 | Val Acc: 0.7792
Fold s1 | Epoch 7 | Train Loss: 0.5826 | Val Loss: 0.9513 | Val Acc: 0.4333
Fold s1 | Epoch 8 | Train Loss: 0.5349 | Val Loss: 1.8020 | Val Acc: 0.3583
Fold s1 | Epoch 9 | Train Loss: 0.5087 | Val Loss: 0.9031 | Val Acc: 0.4792
Fold s1 | Epoch 10 | Train Loss: 0.4794 | Val Loss: 1.3455 | Val Acc: 0.4042
Fold s1 | Epoch 11 | Train Loss: 0.4481 | Val Loss: 0.5372 | Val Acc: 0.6375
Fold s1 | Epoch 12 | Train Loss: 0.4249 | Val Loss: 0.6428 | Val Acc: 0.7167
Fold s1 | Epoch 13 | Train Loss: 0.3989 |

In [15]:
for r in fold_results:
    print(f"Subject: {r['subject']}")
    print(f"Val Loss: {r['val_loss']:.4f}")
    print(f"Accuracy: {r['accuracy']:.4f}")
    print(f"Macro F1: {r['macro_f1']:.4f}")
    print("Confusion Matrix:")
    print(r["confusion_matrix"])
    print("-" * 40)

Subject: s1
Val Loss: 0.5372
Accuracy: 0.6375
Macro F1: 0.6018
Confusion Matrix:
[[73  7  0]
 [67 13  0]
 [ 0 13 67]]
----------------------------------------
Subject: s2
Val Loss: 0.3353
Accuracy: 0.9083
Macro F1: 0.9073
Confusion Matrix:
[[73  7  0]
 [ 4 66 10]
 [ 0  1 79]]
----------------------------------------
Subject: s3
Val Loss: 0.4358
Accuracy: 0.8417
Macro F1: 0.8401
Confusion Matrix:
[[69 11  0]
 [27 53  0]
 [ 0  0 80]]
----------------------------------------
Subject: s4
Val Loss: 0.5415
Accuracy: 0.7000
Macro F1: 0.6374
Confusion Matrix:
[[11 69  0]
 [ 0 80  0]
 [ 0  3 77]]
----------------------------------------
Subject: s5
Val Loss: 0.3748
Accuracy: 0.8292
Macro F1: 0.8157
Confusion Matrix:
[[76  4  0]
 [25 43 12]
 [ 0  0 80]]
----------------------------------------


In [14]:
for s in np.unique(all_subjects):
    mask = np.array(all_subjects) == s
    print(s, np.bincount(y[mask]))

s1 [80 80 80]
s2 [80 80 80]
s3 [80 80 80]
s4 [80 80 80]
s5 [80 80 80]
